# data

In [1]:
categories = set()

def load_data(data_file):
    Data = {}
    with open (data_file, "rt", encoding="utf-8") as f:
        # 文本使用空行进行分割句子
        for idx, line in enumerate(f.read().split("\n\n")):
            if not line:
                break
            sentence, labels = "", []
            for i, item in enumerate(line.split("\n")):
                char, tag = item.split(" ")
                sentence += char
                if tag.startswith("B"):
                    labels.append([i, i, char, tag[2:]])   # Remove the B- or I-
                    categories.add(tag[2:])
                elif tag.startswith("I"):
                    labels[-1][1] = i
                    labels[-1][2] += char
            Data[idx] = {
                "sentence" : sentence,
                "labels" : labels
            }
    return Data

In [2]:
path = "./state3/dataset/ner_data/medical.train"
ds = load_data(path)

In [3]:
for idx, example in ds.items():
    print(idx, example)
    if idx >= 10:
        break

0 {'sentence': '现头昏口苦', 'labels': [[3, 4, '口苦', '临床表现']]}
1 {'sentence': '目的观察复方丁香开胃贴外敷神阙穴治疗慢性心功能不全伴功能性消化不良的临床疗效', 'labels': [[4, 10, '复方丁香开胃贴', '中医治疗'], [20, 32, '心功能不全伴功能性消化不良', '西医诊断']]}
2 {'sentence': '舒肝和胃消痞汤；功能性消化不良', 'labels': [[8, 14, '功能性消化不良', '西医诊断']]}
3 {'sentence': '患者３ａ前咯血，被诊断为肺结核，住院４０余天时出现腹痛，经治疗好转，但时有发作，坚持服抗痨药３ａ后，因腹痛基本缓解，肺结核治愈而停药', 'labels': [[5, 6, '咯血', '临床表现'], [12, 14, '肺结核', '西医诊断'], [58, 60, '肺结核', '西医诊断']]}
4 {'sentence': '治疗组采用复方蜥蜴散不同微粒组合剂（密点麻蜥、炙黄芪、焦乌梅、炒白芍、三七、半枝莲等）治疗', 'labels': [[5, 9, '复方蜥蜴散', '方剂'], [18, 21, '密点麻蜥', '中药'], [23, 25, '炙黄芪', '中药'], [27, 29, '焦乌梅', '中药'], [31, 33, '炒白芍', '中药'], [35, 36, '三七', '中药'], [38, 40, '半枝莲', '中药']]}
5 {'sentence': '经检查诊断为“缩窄性心包炎”', 'labels': [[7, 12, '缩窄性心包炎', '西医诊断']]}
6 {'sentence': '按语：肝硬化腹水属中医“鼓胀”、“水鼓”范畴，鼓胀为临床疑难重症之一，初期多为气滞湿阻致使腹水形成，日久脾肾亦虚，肝肾阴虚，肝脾肾三脏功能失调，气滞血瘀，水饮停留于腹中，本虚而标实', 'labels': [[17, 18, '水鼓', '中医诊断']]}
7 {'sentence': '柴枳半夏泻心汤对功能性消化不良胃动力的影响', 'labels': [[0, 6, '柴枳半夏泻心汤', '方剂'], [8, 14, '功能性消化不良', '西医诊断']]}
8 {'sent

In [4]:
sentence = ds[9]["sentence"]
labels = ds[9]["labels"]

out_list = [{
    "text": text, "type": tag, "start": start, "end": end 
} for start, end, text, tag in labels]
out_list, sentence

([{'text': '气滞胃痛颗粒', 'type': '中医治疗', 'start': 0, 'end': 5},
  {'text': '便秘型肠易激综合征', 'type': '西医诊断', 'start': 13, 'end': 21}],
 '气滞胃痛颗粒联合乳果糖治疗便秘型肠易激综合征临床研究')

In [5]:
import json

def convert_to_sft_format(Data):
    sft_data = []
    for idx, sample in Data.items():
        sentence = sample["sentence"]
        labels = sample["labels"]

        # 构造output
        output = [
            {"text": text, "type": tag, "start": start, "end": end}
            for start, end, text, tag in labels
        ]
        prompt = """
        你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并标注它们的类型、起止位置，并输出标准 JSON。
        """
        sft_data.append({
            "instruction": prompt,
            "input": sentence,
            "output": json.dumps({"entities": output}, ensure_ascii=False)
        })
    return sft_data


In [6]:
ds_json = convert_to_sft_format(ds)
ds_json[:10]

[{'instruction': '\n        你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并标注它们的类型、起止位置，并输出标准 JSON。\n        ',
  'input': '现头昏口苦',
  'output': '{"entities": [{"text": "口苦", "type": "临床表现", "start": 3, "end": 4}]}'},
 {'instruction': '\n        你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并标注它们的类型、起止位置，并输出标准 JSON。\n        ',
  'input': '目的观察复方丁香开胃贴外敷神阙穴治疗慢性心功能不全伴功能性消化不良的临床疗效',
  'output': '{"entities": [{"text": "复方丁香开胃贴", "type": "中医治疗", "start": 4, "end": 10}, {"text": "心功能不全伴功能性消化不良", "type": "西医诊断", "start": 20, "end": 32}]}'},
 {'instruction': '\n        你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并标注它们的类型、起止位置，并输出标准 JSON。\n        ',
  'input': '舒肝和胃消痞汤；功能性消化不良',
  'output': '{"entities": [{"text": "功能性消化不良", "type": "西医诊断", "start": 8, "end": 14}]}'},
 {'instruction': '\n        你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并标注它们的类型、起止位置，并输出标准 JSON。\n        ',
  'input': '患者３ａ前咯血，被诊断为肺结核，住院４０余天时出现腹痛，经治疗好转，但时有发作，坚持服抗痨药３ａ后，因腹痛基本缓解，肺结核治愈而停药',
  'output': '{"entities": [{"text": "咯血", "type": "临床表现", "start": 5, "end": 6

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/qwen2.5-0.5B-instruct")
# model = AutoModelForCausalLM.from_pretrained("Qwen/qwen2.5-0.5B-instruct")
tokenizer.special_tokens_map

{'eos_token': '<|im_end|>',
 'pad_token': '<|endoftext|>',
 'additional_special_tokens': ['<|im_start|>',
  '<|im_end|>',
  '<|object_ref_start|>',
  '<|object_ref_end|>',
  '<|box_start|>',
  '<|box_end|>',
  '<|quad_start|>',
  '<|quad_end|>',
  '<|vision_start|>',
  '<|vision_end|>',
  '<|vision_pad|>',
  '<|image_pad|>',
  '<|video_pad|>']}

In [9]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [10]:
system ="你是一个专业的生物医学信息抽取助手。请从用户提供的句子中识别所有实体，并输出标准 JSON。"
sentence = ds_json[12]["input"]
resp = ds_json[12]["output"]
sentence, resp

('二金汤是胡老自拟的经验方，由羊草结、藿香、木香、鸡内金、郁金、台乌、延胡、川楝子、柴胡、茵陈等约组成',
 '{"entities": [{"text": "羊草结", "type": "中药", "start": 14, "end": 16}, {"text": "木香", "type": "中药", "start": 21, "end": 22}, {"text": "鸡内金", "type": "中药", "start": 24, "end": 26}, {"text": "柴胡", "type": "中药", "start": 41, "end": 42}, {"text": "茵陈", "type": "中药", "start": 44, "end": 45}]}')

In [11]:
message = [
            {"role": "system", "content": system},
            {"role": "ruser", "content": sentence},
            {"role": "assistant", "content": resp},        
    ]

In [12]:
tokened_out =tokenizer.apply_chat_template(
    message,
    tokenize=False, # 这里先不开启
)
tokened_out

'<|im_start|>system\n你是一个专业的生物医学信息抽取助手。请从用户提供的句子中识别所有实体，并输出标准 JSON。<|im_end|>\n<|im_start|>assistant\n{"entities": [{"text": "羊草结", "type": "中药", "start": 14, "end": 16}, {"text": "木香", "type": "中药", "start": 21, "end": 22}, {"text": "鸡内金", "type": "中药", "start": 24, "end": 26}, {"text": "柴胡", "type": "中药", "start": 41, "end": 42}, {"text": "茵陈", "type": "中药", "start": 44, "end": 45}]}<|im_end|>\n'

In [ ]:
def process_func(example):
    max_length = 512

    system_prompt = """你是一个专业的医学信息抽取助手。
    请根据输入句子识别所有的实体，输出 JSON 格式，包含字段：
    text, type, start, end。

    示例：
    句子：气滞胃痛颗粒联合乳果糖治疗便秘型肠易激综合征临床研究
    输出：
    {
    "entities": [
        {"text": "气滞胃痛颗粒", "type": "中医治疗", "start": 0, "end": 5},
        {"text": "便秘型肠易激综合征", "type": "西医诊断", "start": 13, "end": 21},
    ]
    }
    """

    user_prompt = f"{example['instruction']}\n句子：{example['input']}"
    assistant_resp = example["output"]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_resp},
    ]

    # 获取完整的 tokenized
    tokenized_full = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False, # 训练时不需要为生成添加特殊 prompt
        max_length=max_length,
        truncation=True,
        return_tensors=None,
    )

    attention_mask = [1] * len(tokenized_full)
    
    # 获取 system + user 的 tokenized
    tokenized_prefix = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        tokenize=True,
        add_generation_prompt=True, # 添加 <|im_start|>assistant\n 的前缀
        return_tensors=None,
    )

    # Labels: mask 掉 prefix_len 之前的部分
    prefix_len = len(tokenized_prefix)
    labels = [-100] * min(prefix_len, len(tokenized_full)) + tokenized_full[min(prefix_len, len(tokenized_full)):]

    return {
        "input_ids": tokenized_full,
        "attention_mask": attention_mask,
        "labels": labels,
    }


In [48]:
from datasets import Dataset

dataset = Dataset.from_list(ds_json)
dataset[0]

{'instruction': '\n        你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并标注它们的类型、起止位置，并输出标准 JSON。\n        ',
 'input': '现头昏口苦',
 'output': '{"entities": [{"text": "口苦", "type": "临床表现", "start": 3, "end": 4}]}'}

In [49]:
train_data = dataset.map(process_func, remove_columns=dataset.column_names)

Map:   0%|          | 0/5259 [00:00<?, ? examples/s]

In [50]:
print(train_data, train_data[0])

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 5259
}) {'input_ids': [151644, 8948, 198, 56568, 101909, 104715, 104316, 27369, 118797, 110498, 8997, 262, 220, 14880, 100345, 31196, 109949, 102450, 104152, 101565, 3837, 66017, 4718, 51461, 120, 28330, 3837, 102298, 44931, 28311, 262, 1467, 11, 943, 11, 1191, 11, 835, 3407, 262, 80426, 28311, 262, 26853, 98, 44729, 5122, 99180, 101742, 100518, 100406, 107561, 101101, 100489, 27773, 100443, 101899, 111795, 24300, 100859, 86744, 99591, 118458, 104595, 99556, 198, 262, 70568, 28311, 262, 341, 262, 330, 10499, 788, 2278, 286, 5212, 1318, 788, 330, 99180, 101742, 100518, 100406, 107561, 497, 330, 1313, 788, 330, 104823, 101899, 497, 330, 2468, 788, 220, 15, 11, 330, 408, 788, 220, 20, 1583, 286, 5212, 1318, 788, 330, 111795, 24300, 100859, 86744, 99591, 118458, 497, 330, 1313, 788, 330, 119012, 105262, 497, 330, 2468, 788, 220, 16, 18, 11, 330, 408, 788, 220, 17, 16, 1583, 262, 5133, 262, 456, 257, 151645, 19

In [59]:
print(tokenizer.decode(train_data[95]["input_ids"], skip_special_tokens=False))

<|im_start|>system
你是一个专业的医学信息抽取助手。
    请根据输入句子识别所有的实体，输出 JSON 格式，包含字段：
    text, type, start, end。

    示例：
    句子：气滞胃痛颗粒联合乳果糖治疗便秘型肠易激综合征临床研究
    输出：
    {
    "entities": [
        {"text": "气滞胃痛颗粒", "type": "中医治疗", "start": 0, "end": 5},
        {"text": "便秘型肠易激综合征", "type": "西医诊断", "start": 13, "end": 21},
    ]
    }
    <|im_end|>
<|im_start|>user

        你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并标注它们的类型、起止位置，并输出标准 JSON。
        
句子：结论电针配合雷贝拉唑治疗胃食管反流病疗效好，可明显促进ＧＡＳ的分泌<|im_end|>
<|im_start|>assistant
{"entities": [{"text": "电针", "type": "中医治疗", "start": 2, "end": 3}, {"text": "胃食管反流病", "type": "西医诊断", "start": 12, "end": 17}]}<|im_end|>

